# Mens-in-de-Lus: Pre-Actie Poorten, Risiconiveaus en Audit Logging

De README voor deze les introduceert Mens-in-de-Lus met een korte snippet die de gebruiker vraagt om `GOEDGEKEURD` of `AFGEWEZEN` te antwoorden nadat de agent al een reactie heeft geproduceerd. Dat patroon is een prima uitgangspunt, maar productieve HITL-implementaties hebben doorgaans drie extra onderdelen nodig:

1. Een **pre-actie poort** die **voor** de uitvoering van een risicovolle stap door de agent werkt, zodat kosten, onomkeerbaarheid en latentie onder controle blijven.
2. **Risiconiveaus**, zodat acties met laag risico automatisch worden uitgevoerd, acties met gemiddeld risico batch-goedgekeurd worden, en alleen acties met hoog risico door een mens worden geblokkeerd.
3. Een **auditlog plus revisielus**, zodat elke poortbeslissing wordt vastgelegd als JSONL, en een afwijzing de agent heractiveert met een gestructureerde reden in plaats van alleen `Bezig met herzien...` te tonen.

Deze notebook bouwt elk van deze functies bovenop dezelfde primitieven als `06-system-message-framework.ipynb`. Hij draait end-to-end in `DEMO_MODE = True` (geen interactieve invoer nodig) of met echte `input()` prompts wanneer `DEMO_MODE = False`. Opmerking: in DEMO_MODE is de herhalingspoging van het derde doel gescript zodat de lusmechanica end-to-end zichtbaar is. Echte herzieningsgestuurde herclassificatie vereist `DEMO_MODE = False` en een operator.

**Buiten scope (wordt behandeld in andere lessen):** authenticatie en toegangscontrole (Les 06 README bedreiging 2), tool-call middleware (Les 14 MAF deep dive), multi-agent debatpatronen.


In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import OpenAI

load_dotenv()

DEMO_MODE = True  # set False to use real input() prompts

# Per-run unique log filename so demo runs don't overwrite each other and
# the notebook doesn't touch any pre-existing gate_log.jsonl in the working
# directory.
GATE_LOG_PATH = Path(
    f"gate_log_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.jsonl"
)

# This notebook uses the Azure OpenAI Responses API via the stable /openai/v1/ endpoint.
# GitHub Models is deprecated (retiring July 2026) and does not support the Responses API.
endpoint = os.environ.get("AZURE_OPENAI_ENDPOINT", "")
if not endpoint:
    raise RuntimeError(
        "AZURE_OPENAI_ENDPOINT environment variable is not set. This notebook needs "
        "an Azure OpenAI resource with a model deployment that supports the Responses "
        "API. Set AZURE_OPENAI_ENDPOINT and AZURE_OPENAI_DEPLOYMENT in "
        "your environment or a local .env file, then run `az login`."
    )

deployment = os.environ["AZURE_OPENAI_DEPLOYMENT"]

# Authenticate with Entra ID (run `az login` first). No api_version is needed.
token_provider = get_bearer_token_provider(
    DefaultAzureCredential(),
    "https://cognitiveservices.azure.com/.default",
)

client = OpenAI(
    base_url=f"{endpoint.rstrip('/')}/openai/v1/",
    api_key=token_provider,
)



## Patroon 1: Pre-actie poort

De HITL-fragment uit de README roept de agent eerst op, en vraagt vervolgens de gebruiker om de output goed te keuren. Dat is een **post-actie** stroom. De agent heeft al uitgevoerd, dus de LLM-aanroep kosten zijn al betaald, en elk neveneffect (verzonden e-mail, geschreven database regel, geplaatste reactie) heeft al plaatsgevonden.

Een **pre-actie** stroom plaatst de poort vóórdat de agent de risicovolle stap uitvoert. De agent stelt de actie voor, de poort besluit of deze uitgevoerd wordt, en alleen bij goedkeuring vindt het neveneffect plaats.

| Aspect | Post-actie goedkeuring (README-fragment) | Pre-actie poort (deze notebook) |
|---|---|---|
| Wanneer wordt goedkeuring uitgevoerd? | Nadat de agent output heeft geproduceerd | Voordat een neveneffect wordt uitgevoerd |
| LLM-kosten bij afwijzing | Al betaald | Alleen betaald voor het voorstel, niet de actie |
| Onomkeerbare neveneffecten | Mogelijk (de actie is al gebeurd) | Voorkomen |
| Audit duidelijkheid | Goedkeuring is een printstatement | Goedkeuring is een JSON-record met tijdstempel, actie, reden |


In [ ]:
def gate_action(action_description: str, risk_tier: str, attempt: int = 0) -> dict:
    """Run a single pre-action gate.

    Returns a decision dict with keys: decision, reason, ts.
    Decision is one of: approve, deny, escalate.
    Safe default on EOF or unexpected input is deny.

    DEMO_MODE behavior: high-risk actions are denied on attempt 0 and
    auto-approved on attempt >= 1. This is scripted approval to show the
    loop mechanics (deny -> retry -> approve). It is NOT revision-driven
    re-classification. Real revision-driven re-classification requires
    DEMO_MODE=False and a human operator who evaluates the revised
    proposal on its own merits.
    """
    print(f"[gate] proposed action ({risk_tier}, attempt={attempt}): {action_description}")

    if DEMO_MODE:
        if risk_tier == "high":
            decision = "approve" if attempt >= 1 else "deny"
            reason = (
                "DEMO_MODE: scripted approval on retry to show loop mechanics"
                if attempt >= 1
                else "DEMO_MODE: high risk denied on first attempt"
            )
        else:
            decision = "approve"
            reason = f"DEMO_MODE canned response for tier={risk_tier}"
    else:
        try:
            raw = input("[gate] approve / deny / escalate? ").strip().lower()
        except EOFError:
            raw = ""
        if raw in {"approve", "deny", "escalate"}:
            decision, reason = raw, "operator input"
        elif raw == "":
            decision, reason = "deny", "no input received, defaulted to deny"
        else:
            decision, reason = "deny", f"invalid input {raw!r}, defaulted to deny"

    return {
        "decision": decision,
        "reason": reason,
        "action": action_description,
        "risk_tier": risk_tier,
        "ts": datetime.now(timezone.utc).isoformat(),
    }


## Patroon 2: Risiconiveaus

Niet elke actie heeft menselijke goedkeuring nodig. Een alleen-lezen opzoeking via een openbare API heeft andere belangen dan het versturen van een klantmail. Beide op dezelfde manier behandelen verspilt de aandacht van de operator en vertraagt de agent.

Een eenvoudig 3-niveaus model:

| Niveau | Voorbeelden | Goedkeuringsproces |
|---|---|---|
| `laag` (alleen lezen) | Zoeken in een kennisbank, vluchtopties opzoeken, een openbare webpagina ophalen | Automatisch uitvoeren, gelogd voor audit |
| `middel` (goedkope mutatie) | Een resultaat cachen, een teller verhogen, een herinnering plannen | Automatisch uitvoeren, maar dagelijks in batch beoordelen |
| `hoog` (extern gericht of onomkeerbaar) | Een e-mail versturen, een kaart belasten, posten op een openbaar kanaal | Blokkeren tot menselijke goedkeuring |

Dit is één tiering. Productiesystemen gebruiken vaak fijnere niveaus (bijv. AWS IAM permissieniveaus, rolgebaseerde toegangsniveaus). De 3-niveaus versie hieronder is de kleinste bruikbare versie voor een agent die lezen en acties met bijwerkingen mengt.

De onderstaande classifier gebruikt sleutelwoordheuristieken zodat de demo deterministisch en goedkoop blijft. In een productiesysteem zou je dit vervangen door een geleerde classifier of een beleidsengine.


In [ ]:
LOW_RISK_KEYWORDS = {
    "look", "lookup", "search", "fetch", "read", "query", "view",
    "get", "list", "weather", "summarize",
}
HIGH_RISK_KEYWORDS = {
    "send", "email", "post", "publish", "charge", "pay", "transfer",
    "delete", "drop", "cancel", "refund",
}
MEDIUM_RISK_KEYWORDS = {
    "cache", "schedule", "reminder", "book", "reserve", "update",
    "increment", "log",
}

AUTO_APPROVE_REASONS = {
    "low": "auto-approved (low risk)",
    "medium": "auto-approved (medium risk, queued for batched review)",
}


def classify_risk(action: str) -> str:
    """Classify an action string into one of: low, medium, high.

    Keyword-based heuristic. Checks high-risk first (most severe), then
    low-risk explicit reads, then medium-risk mutations. Unrecognized
    actions default to medium, not low.

    Default for unrecognized actions is 'medium', not 'low'. A read-only
    keyword set will always have blind spots, and the parent README's
    threat list (critical-system access, knowledge-base poisoning,
    cascading errors) all involve cases an action-name alone cannot rule
    out. Routing unknown actions through batched review is the safer
    default than auto-executing them.
    """
    text = action.lower()
    if any(kw in text for kw in HIGH_RISK_KEYWORDS):
        return "high"
    if any(kw in text for kw in LOW_RISK_KEYWORDS):
        return "low"
    if any(kw in text for kw in MEDIUM_RISK_KEYWORDS):
        return "medium"
    # Explicit fail-safe default: unrecognized actions route to batched review.
    return "medium"


def tiered_gate(action: str, attempt: int = 0) -> dict:
    """Classify then gate. Low and medium tiers auto-approve; high blocks."""
    tier = classify_risk(action)
    if tier in AUTO_APPROVE_REASONS:
        return {
            "decision": "approve",
            "reason": AUTO_APPROVE_REASONS[tier],
            "action": action,
            "risk_tier": tier,
            "ts": datetime.now(timezone.utc).isoformat(),
        }
    return gate_action(action, tier, attempt=attempt)


## Patroon 3: Auditlogboek en revisielus

Een `print("Response approved.")` is geen auditlogboek. Voor vertrouwen moet elke poortbeslissing worden vastgelegd als een gestructureerd evenement dat je later kunt opvragen, afspelen of koppelen aan een incidentreview.

Twee onderdelen:

1. **Alleen toevoegen JSONL.** Eén regel per beslissing, met tijdstempel, actie, laag, beslissing, reden. Gemakkelijk te doorzoeken, gemakkelijk later naar een echte logstore te verzenden.
2. **Revisielus bij afwijzing.** Wanneer de poort `deny` teruggeeft, geeft de agent zichzelf opnieuw een prompt met de afwijzingsreden in context, zodat het volgende voorstel het probleem kan vermijden.


In [ ]:
def log_decision(decision: dict) -> None:
    """Append a gate decision to the JSONL audit log."""
    with GATE_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(decision) + "\n")


def propose_action(goal: str, prior_rejection: str | None = None) -> str:
    """Ask the LLM to propose a concrete next action for a goal.

    If prior_rejection is provided, it is fed back so the LLM can avoid
    the same failure mode in the next proposal.
    """
    system = (
        "You are an action planner for an agent. Propose ONE concrete next\n"
        "action (a single sentence) toward the user's goal. If a prior\n"
        "rejection reason is given, propose a different action that addresses\n"
        "the rejection."
    )
    user_text = f"Goal: {goal}"
    if prior_rejection:
        user_text += f"\n\nPrior proposal was denied. Reason: {prior_rejection}"

    response = client.responses.create(
        model=deployment,
        input=[
            {"role": "system", "content": system},
            {"role": "user", "content": user_text},
        ],
        store=False,
    )
    return response.output_text.strip()


def run_with_revision(goal: str, max_revisions: int = 2) -> dict:
    """Propose, gate, and on rejection revise up to max_revisions times."""
    prior_reason: str | None = None
    for attempt in range(max_revisions + 1):
        action = propose_action(goal, prior_rejection=prior_reason)
        decision = tiered_gate(action, attempt=attempt)
        decision["attempt"] = attempt
        log_decision(decision)
        if decision["decision"] == "approve":
            return decision
        prior_reason = decision["reason"]
    return {**decision, "final": "max_revisions_reached"}



In [ ]:
# End-to-end demo: three goals at three different risk profiles.
# GATE_LOG_PATH is per-run (timestamped) so no prior log is touched.

goals = [
    "Look up the weather in Seattle for the customer's trip planning.",
    "Schedule a reminder for the customer to check in 24 hours before their flight.",
    "Send a marketing email to the customer about premium upgrade options.",
]

for goal in goals:
    print(f"\n=== Goal: {goal} ===")
    outcome = run_with_revision(goal, max_revisions=1)
    print(f"[final] {outcome['decision']} ({outcome['reason']})")

print(f"\n=== Audit log ({GATE_LOG_PATH.name}) ===")
for line in GATE_LOG_PATH.read_text(encoding="utf-8").splitlines():
    record = json.loads(line)
    print(f"  [{record['risk_tier']:6s}] {record['decision']:8s} "
          f"attempt={record.get('attempt', '?')} action={record['action'][:140]}")


## Aanvullende bronnen

Verschillende andere openbare projecten implementeren variaties van deze HITL-patronen. Vergelijk benaderingen om te vinden wat past bij jouw stack:

- **LangChain** human-in-the-loop tool wrappers ([docs](https://python.langchain.com/docs/integrations/tools/human_tools)): inzetbare tool wrappers die de uitvoering pauzeren voor menselijke invoer.
- **AutoGen** `UserProxyAgent` ([v0.2 docs](https://microsoft.github.io/autogen/0.2/docs/topics/human-in-the-loop); AutoGen v0.4+ heeft dit herstructureerd): gebruikt een agentrol specifiek om de mens te vertegenwoordigen in multi-agent gesprekken.
- **Microsoft Agent Framework (MAF)** function-invocation middleware ([docs](https://learn.microsoft.com/agent-framework/)): middleware die rondom elke tool/function call draait, geschikt voor gate-logica en goedkeuringsstromen.

Elk project behandelt de drie subpatronen op een andere manier: LangChain wikkelt ze in als tools, AutoGen gebruikt een agentrol, en Microsoft Agent Framework gebruikt function-invocation middleware. Lees één of twee implementaties van begin tot eind voordat je een ontwerp kiest voor je eigen agent.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Dit document is vertaald met behulp van de AI vertaaldienst [Co-op Translator](https://github.com/Azure/co-op-translator). Hoewel we streven naar nauwkeurigheid, dient u er rekening mee te houden dat geautomatiseerde vertalingen fouten of onnauwkeurigheden kunnen bevatten. Het originele document in de oorspronkelijke taal moet worden beschouwd als de gezaghebbende bron. Voor kritieke informatie wordt professionele menselijke vertaling aanbevolen. Wij zijn niet aansprakelijk voor eventuele misverstanden of verkeerde interpretaties die voortvloeien uit het gebruik van deze vertaling.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
